# Training 03 · Baseline predictor: pgd pairs

R0 fine-tuned once on preference pairs made by waveform-PGD pairs: the input is preferred over a white-box adversarial copy. Same patch losses and the same step budget as three
patch-loop rounds; starts from the **released** R0 of the same seed. 3 seeds (`baseline_pgd_s0..2`),
about 4 h each.

In [1]:
import os, sys
from pathlib import Path

# Find the repository root from wherever the notebook runs (notebooks/ or a subfolder).
ROOT = Path.cwd().resolve()
while not (ROOT / "src" / "hackpatch").is_dir():
    if ROOT.parent == ROOT:
        raise RuntimeError("start the notebook from inside the repository")
    ROOT = ROOT.parent
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

import pandas as pd
from IPython.display import Markdown, display
from hackpatch import repro

pd.set_option("display.width", 200, "display.max_columns", 40, "display.max_rows", 200)

In [2]:
# Parameters
# LAUNCH = False only shows what would run (the job script of the first run); True starts the training.
LAUNCH = os.environ.get("NB_LAUNCH", "0") == "1"
# RUN = "slurm": one GPU job per run (sbatch); "local": in this kernel (needs a GPU, runs sequentially).
RUN = os.environ.get("NB_RUN", "slurm")

In [3]:
reg = repro.registry()
KEYS = [k for k, v in reg.items() if v["kind"] in ['baseline'] and k.startswith("baseline_pgd")]
print(len(KEYS), "runs:", ", ".join(KEYS))

3 runs: baseline_pgd_s0, baseline_pgd_s1, baseline_pgd_s2


In [4]:
# Stored configuration of the first run (checkpoint paths point at the local released copies).
from omegaconf import OmegaConf
print(OmegaConf.to_yaml(repro.load_config(KEYS[0])))

config_name: baseline_pgd
task: train_predictor
phase: main
seed: 0
run_name: baseline_pgd_s0__2026-09-15_16-10-55
init_ckpt: /fs/scratch/PAS2301/alialavi/cache/audio_assessment_hackability_release/checkpoints/r0_s0.checkpoints.best.pt
model:
  pretrained: facebook/wav2vec2-base
  freeze_feature_encoder: true
  freeze_encoder: false
data:
  corpus: bvcc
  max_seconds: 6.0
  num_workers: 8
train:
  pair_source: pgd
  steps: 4500
  batch_size: 8
  pair_batch_size: 8
  lr_encoder: 1.0e-05
  lr_head: 0.0001
  weight_decay: 0.01
  warmup_steps: 100
  cosine: false
  grad_clip: 1.0
  w_mse: 1.0
  w_rank: 1.0
  margin: 0.5
  noise_snr_db:
  - 5.0
  - 30.0
  pgd:
    eps: 0.002
    steps: 5
    step_size: 0.0005
  log_every: 25
  eval_every: 500
  select: last
  pairs_csv: null
  val_pairs_csv: null
eval:
  batch_size: 16
  max_seconds: 20.0
  n_boot: 1000
wandb:
  entity: alialavi
  mode: online
  group: null



In [5]:
if LAUNCH:
    repro.run_keys(KEYS, "retrain", RUN)  # outputs: {scratch}/projects/audio_assessment_hackability/retrain/<key>/
else:
    print("LAUNCH=False: nothing started. Job script for", KEYS[0], "->")
    repro.submit(KEYS[0], "retrain", dry_run=True)

LAUNCH=False: nothing started. Job script for baseline_pgd_s0 ->
#!/bin/bash
#SBATCH --account=PAS2966
#SBATCH --partition=nextgen
#SBATCH --job-name=retrain_baseline_pgd_s0
#SBATCH --output=/fs/scratch/PAS2301/alialavi/projects/audio_assessment_hackability/retrain/_logs/%x_%j.out
#SBATCH --time=02:00:00
#SBATCH --nodes=1
#SBATCH --ntasks-per-node=1
#SBATCH --cpus-per-task=12
#SBATCH --mem=64G
#SBATCH --gpus-per-node=1

set -euo pipefail
cd /users/PAS2301/alialavi/projects/audio_assessment_hackability
source scripts/env.sh
python scripts/repro_run.py --mode retrain --key baseline_pgd_s0



## Retrained vs original

Agreement with human ratings on BVCC test for each finished retrained predictor, next to the paper's value for
the same run (`analysis/out/e2_long.csv`). Needs a GPU in this kernel (about a minute per predictor).

In [6]:
import torch
from hackpatch.data.manifests import read_manifest
from hackpatch.eval.correlation import correlation_report
from hackpatch.models.ssl_mos import load_predictor
from hackpatch.patch.trainer import predict_manifest

paper = pd.read_csv(ROOT / "analysis/out/e2_long.csv").query("corpus == 'bvcc' and split == 'test'")
rows = []
if torch.cuda.is_available():
    test = read_manifest("bvcc", "test")
    for k in KEYS:
        sel = repro.RUNS / "retrain" / k / "results" / "selected_checkpoint.txt"
        if not sel.exists():
            continue
        seed = int(k.rsplit("_s", 1)[1])
        name = "R0" if k.startswith("r0") else k.rsplit("_s", 1)[0]
        df = test.copy()
        df["pred"] = predict_manifest(load_predictor(sel.read_text().strip()), df, 16, 20.0)
        rep = correlation_report(df, n_boot=100, seed=seed)
        ref = paper[(paper.predictor == name) & (paper.seed == seed)]
        rows.append({"run": k, "sys_srcc": rep["sys_srcc"], "paper_sys_srcc": ref.sys_srcc.mean(),
                     "utt_mse": rep["utt_mse"], "paper_utt_mse": ref.utt_mse.mean()})
pd.DataFrame(rows) if rows else "no GPU in this kernel, or no finished retrained predictor yet"

'no GPU in this kernel, or no finished retrained predictor yet'